# σ-recovery on a synthetic icosphere

End-to-end reproducibility demo for the σ-resolution analysis published
in the **pulse-ep-decay** method paper.

Steps:
1. Build a 642-vertex icosphere (no external mesh file required).
2. Pick a true origin and compute Heat-Method geodesic distances from it.
3. Inject a synthetic Gaussian score field with a known **σ₀**.
4. Re-detect the origin (`argmax(score)`) and re-fit σ̂ on the binned
   radial profile — exactly the pipeline used in the clinical analysis.
5. Confirm σ̂ ≈ σ₀.

A successful run prints a recovered σ̂ within a few percent of the
injected σ₀ and produces two figures: the score field on the sphere
and the radial profile with the fitted curve.

## 1. Mesh + helpers

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# allow importing the local _mesh_utils.py helper
sys.path.insert(0, str(Path.cwd()))
from _mesh_utils import icosphere

from pulse_ep_decay import (
    MeshHeatSolver,
    fit_gaussian_decay,
    gaussian_decay,
    generate_synthetic_field,
    recover_sigma,
    run_sigma_recovery,
)

vertices, triangles = icosphere(subdivisions=2, radius_mm=25.0)
print(f'mesh: {vertices.shape[0]} vertices, {triangles.shape[0]} triangles')

## 2. Inject a synthetic field

We pick the vertex closest to (1, 0, 0)·R as the *true* origin,
compute Heat-Method geodesics from it, and use those distances
to draw a noisy Gaussian field with σ₀ = 8 mm.

In [ ]:
SIGMA_TRUE_MM = 8.0
NOISE_LEVEL = 0.05  # 5 % of amplitude
RNG = np.random.default_rng(seed=42)

# Heat-Method solver: factorises the Laplacian once, reusable.
solver = MeshHeatSolver(vertices, triangles)

# True origin: vertex closest to +x pole
true_origin = int(np.argmax(vertices[:, 0]))
geo_dist_true = solver.distances(true_origin)
print(f'true origin = vertex {true_origin}, '
      f'{np.sum(np.isfinite(geo_dist_true))}/{len(geo_dist_true)} reachable')

scores = generate_synthetic_field(
    geo_dist_true,
    sigma_true_mm=SIGMA_TRUE_MM,
    noise_level=NOISE_LEVEL,
    missing_fraction=0.05,
    rng=RNG,
)
print(f'scores: min={np.nanmin(scores):.1f}  max={np.nanmax(scores):.1f}  '
      f'NaN={np.isnan(scores).sum()}/{len(scores)}')

## 3. Visualise the field on the mesh

In [ ]:
import pyvista as pv
pv.set_jupyter_backend('trame')  # use 'static' on headless servers

faces_flat = np.column_stack([np.full(len(triangles), 3, dtype=np.int64), triangles]).ravel()
mesh = pv.PolyData(vertices, faces_flat)
mesh['score'] = scores

pl = pv.Plotter(notebook=True)
pl.add_mesh(mesh, scalars='score', nan_color='lightgrey', cmap='turbo', smooth_shading=True)
pl.add_text(f'Synthetic Gaussian field — σ₀ = {SIGMA_TRUE_MM:.1f} mm', position='upper_left')
pl.show()

## 4. Recover σ̂

`recover_sigma` is the published pipeline:

1. `argmax(score)` → origin (will move under noise — captures the
   *origin-misplacement* bias as well as the fit bias).
2. Heat-Method geodesics from that origin.
3. Bin distances into 2 mm rings, average scores per bin, fit
   `A·exp(-d²/(2σ²)) + B` with `scipy.optimize.curve_fit`.

In [ ]:
fit = recover_sigma(vertices, triangles, scores, method='heat', bin_width_mm=2.0)
if fit is None:
    raise RuntimeError('Gaussian fit did not converge — try a larger σ₀ or less noise.')

print(f'σ₀ (injected) = {SIGMA_TRUE_MM:.3f} mm')
print(f'σ̂ (recovered) = {fit.sigma_mm:.3f} mm')
print(f'bias          = {fit.sigma_mm - SIGMA_TRUE_MM:+.3f} mm '
      f'({(fit.sigma_mm - SIGMA_TRUE_MM) / SIGMA_TRUE_MM * 100:+.1f} %)')
print(f'R² (binned)   = {fit.r_squared:.4f}')
print(f'n bins        = {fit.n_bins}')

## 5. Radial profile with fitted curve

In [ ]:
d_grid = np.linspace(0.0, fit.distance_bin_centres_mm.max(), 200)
s_hat = gaussian_decay(d_grid, fit.amplitude, fit.sigma_mm, fit.baseline)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(fit.distance_bin_centres_mm, fit.binned_scores,
           color='#1f77b4', label='binned profile')
ax.plot(d_grid, s_hat, color='#e34a33', linewidth=2,
        label=f'fit: σ̂ = {fit.sigma_mm:.2f} mm')
ax.axvline(SIGMA_TRUE_MM, color='grey', linestyle='--',
           label=f'σ₀ = {SIGMA_TRUE_MM:.2f} mm')
ax.set_xlabel('Geodesic distance from detected origin [mm]')
ax.set_ylabel('Score (binned mean)')
ax.set_title('σ-recovery on synthetic icosphere')
ax.legend()
fig.tight_layout()
fig.show()

## 6. Mini parameter sweep

Same setup, two σ₀ values × two noise levels × 5 trials each. Shows
the distribution of `σ̂ − σ₀` — the kind of plot Figure 3 of the
method paper aggregates.

In [ ]:
results = run_sigma_recovery(
    vertices, triangles,
    sigma_true_mm=[5.0, 8.0],
    noise_levels=[0.0, 0.10],
    missing_fraction=0.05,
    n_trials=5,
    method='heat',
    seed=42,
)

import pandas as pd
df = pd.DataFrame([
    {'sigma_true': r.sigma_true_mm,
     'noise':      r.noise_level,
     'bias_mm':    r.bias_mm,
     'rel_bias_%': r.relative_bias_pct}
    for r in results if r.bias_mm is not None
])
df.groupby(['sigma_true', 'noise']).agg(
    n=('bias_mm', 'size'),
    mean_bias_mm=('bias_mm', 'mean'),
    std_bias_mm=('bias_mm', 'std'),
    mean_rel_bias=('rel_bias_%', 'mean'),
)

## Reproducibility checklist

- Mesh deterministic (icosphere, no I/O).
- Noise RNG seeded (`seed=42`).
- Heat-Method solver factorises the Laplacian once — the result is
  the same regardless of how many times you re-run the cell.
- For an independent **language** check, see [`r/`](r/) — same
  geodesic profile exported to CSV, fit with R `nls()`.